# 07 — Case-study predictions & stats for W3

Reuses the model trained by `06_synth_and_tournament__allie_plus_4features.ipynb`
(input dim 1027 = 1024 Allie embedding + 3 tabular features: `sf15_match`,
`player_elo_z`, `opponent_elo_z`). Loads its weights, re-runs predictions for
(a) the trained detector and (b) a rule-based **simple detector** whose
prediction is literally `move == move_stockfish_15`.

Computes two families of statistics to answer reviewer W3:

**Q1 — the detector does NOT exploit the `top-1 engine == cheat` artifact.**
- headline Macro-F1, cheat P/R/F1 for both detectors on synth-test and tournament;
- agreement / disagreement contingency (model vs simple baseline, split by truth);
- per-cheat-source recall on synth (baseline bombs on lc0, model should hold up);
- conditional slices on synth: recall on cheats where `sf15_match = 0` (baseline misses),
  FP rate on humans where `sf15_match = 1` (baseline cries wolf).

**Q2 — the detector works on a pattern absent from synth: consecutive cheat moves.**
- per-row annotations on tournament: `cheat_run_id`, `cheat_run_pos`,
  `cheat_run_len`, `cheat_order_in_game`;
- recall of the trained detector and the simple baseline bucketed by `cheat_run_pos`,
  `cheat_order_in_game`, `cheat_run_len`.

**Examples for manual inspection** — four pre-filtered parquet files so you can
cherry-pick concrete rows (each row carries `game_id, half_move, fen_before,
move_uci, move_stockfish_15, move_played, prob_model, pred_model, pred_simple,
cheat_run_*`). Parquets live under
`reports/move_level/07_case_study/`. The full enriched tournament &
synth-test predictions are also saved so you can slice further.

In [ ]:
# Cell 1. Imports, seeds, device, paths, run_dir.
import json
import logging
import os
import platform
import socket
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

_REPO_ROOT = Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip())
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from experiments.move_level.configs.dataset_config import DatasetConfig
from experiments.move_level.configs.model_config import ModelConfig
from experiments.move_level.features.feature_config import FeatureConfig
from experiments.move_level.data.synth_dataset import load_synth_data
from experiments.move_level.data.tournament_dataset import load_tournament_data
from experiments.move_level.evaluation.synth_eval import evaluate_on_synth_test
from experiments.move_level.evaluation.tournament_eval import (
    evaluate_on_tournament, metrics_to_long_df,
)
from experiments.move_level.training.metrics import compute_metrics
from experiments.move_level.utils.paths import (
    synth_csv_path, synth_emb_allie_path,
    tournament_csv_path, tournament_emb_allie_path,
)
from experiments.move_level.utils.device import get_device
from experiments.move_level.utils.random import set_all_seeds

SEED = 42
set_all_seeds(SEED)
device = get_device()

SOURCE_RUN_TAG = "06_allie_plus_4features"
source_run_dir = _REPO_ROOT / "reports" / "move_level" / SOURCE_RUN_TAG
source_model_dir = source_run_dir / "models" / "ALL" / "mlp_002"
source_model_path = source_model_dir / "model.pt"
source_meta_path = source_model_dir / "meta.json"

RUN_TAG = "07_case_study"
run_dir = _REPO_ROOT / "reports" / "move_level" / RUN_TAG
(run_dir / "tables").mkdir(parents=True, exist_ok=True)
(run_dir / "predictions").mkdir(parents=True, exist_ok=True)
(run_dir / "examples").mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 1b. Logging setup (file + stdout) and header.
log = logging.getLogger("move_level.casestudy")
log.setLevel(logging.DEBUG)
for h in list(log.handlers):
    log.removeHandler(h)
_fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%Y-%m-%d %H:%M:%S")
_fh = logging.FileHandler(run_dir / "case_study.log", mode="w", encoding="utf-8")
_fh.setLevel(logging.DEBUG); _fh.setFormatter(_fmt); log.addHandler(_fh)
_sh = logging.StreamHandler(sys.stdout)
_sh.setLevel(logging.INFO); _sh.setFormatter(_fmt); log.addHandler(_sh)

def _git_rev():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    except Exception:
        return "?"

log.info("=" * 78)
log.info("RUN TAG        : %s", RUN_TAG)
log.info("source run     : %s", source_run_dir)
log.info("started at     : %s", datetime.now().isoformat(timespec="seconds"))
log.info("hostname       : %s", socket.gethostname())
log.info("platform       : %s", platform.platform())
log.info("python / torch : %s / %s", sys.version.split()[0], torch.__version__)
log.info("device         : %s", device)
log.info("git rev        : %s  |  run dir: %s", _git_rev(), run_dir)
log.info("=" * 78)

In [ ]:
# ------------------------------------------------------------
# Cell 2. Load meta.json from notebook 06 and rebuild configs.
# ------------------------------------------------------------

if not source_model_path.exists():
    raise FileNotFoundError(
        f"Model weights not found: {source_model_path}. "
        f"Run notebook 06 first."
    )
if not source_meta_path.exists():
    raise FileNotFoundError(
        f"Meta not found: {source_meta_path}. Run notebook 06 first."
    )

meta = json.loads(source_meta_path.read_text())
for k in ["feature_stats", "feature_config", "enabled_features", "cheat_sources",
         "best_thr", "input_dim", "model_config"]:
    if k not in meta:
        raise KeyError(f"meta.json is missing required key {k!r}. "
                       f"Re-run notebook 06 with the updated trainer/notebook.")

feature_stats = meta["feature_stats"]
threshold = float(meta["best_thr"])
input_dim = int(meta["input_dim"])
model_cfg = ModelConfig(**meta["model_config"])
feat_cfg = FeatureConfig(**meta["feature_config"])
CHEAT_SOURCES = list(meta["cheat_sources"])
enabled = list(meta["enabled_features"])

log.info("source best_epoch=%s best_val_loss=%.4f best_val_metric=%.4f best_thr=%.3f",
         meta.get("best_epoch"), float(meta.get("best_val_loss", float("nan"))),
         float(meta.get("best_val_metric", float("nan"))), threshold)
log.info("feature_stats       : %s", feature_stats)
log.info("enabled_features    : %s", enabled)
log.info("cheat_sources       : %s", CHEAT_SOURCES)
log.info("ModelConfig         : %s", model_cfg)
log.info("FeatureConfig       : %s", feat_cfg)

In [ ]:
# ------------------------------------------------------------
# Cell 3. Rebuild DatasetConfig and load synth + tournament.
#   Uses training-time feature_stats so standardisation matches exactly.
# ------------------------------------------------------------

N_RATING_BINS = 6
cheat_model_to_bin_idxs = {name: list(range(N_RATING_BINS)) for name in CHEAT_SOURCES}

p_synth_csv = os.environ.get("MOVE_LEVEL_SYNTH_CSV", str(synth_csv_path(_REPO_ROOT)))
p_synth_npz = os.environ.get("MOVE_LEVEL_SYNTH_NPZ", str(synth_emb_allie_path(_REPO_ROOT)))
p_tourn_csv = os.environ.get("MOVE_LEVEL_TOURN_CSV", str(tournament_csv_path(_REPO_ROOT)))
p_tourn_npz = os.environ.get("MOVE_LEVEL_TOURN_NPZ", str(tournament_emb_allie_path(_REPO_ROOT)))
for p, label in [(p_synth_csv, "synth_csv"), (p_synth_npz, "synth_npz"),
                 (p_tourn_csv, "tourn_csv"), (p_tourn_npz, "tourn_npz")]:
    marker = "OK " if Path(p).exists() else "MISS"
    log.info("path %s %-10s %s", marker, label, p)

ds_cfg = DatasetConfig(
    synth_csv_path=p_synth_csv,
    synth_emb_npz_path=p_synth_npz,
    tournament_csv_path=p_tourn_csv,
    tournament_emb_npz_path=p_tourn_npz,
    cheat_models=CHEAT_SOURCES,
    cheat_model_to_bin_idxs=cheat_model_to_bin_idxs,
    build_all_mixture=True,
    batch_size=1024,
    num_workers=0,
    seed=SEED,
    emb_key_human="move_uci",
)

log.info("Loading synth…")
synth_data = load_synth_data(ds_cfg, feat_cfg, feature_stats=feature_stats)
log.info("synth: N=%d input_dim=%d test=%d",
         synth_data.vec_human.shape[0], synth_data.input_dim, len(synth_data.test_idx))
assert synth_data.input_dim == input_dim, (
    f"input_dim mismatch: synth={synth_data.input_dim} meta={input_dim}"
)

log.info("Loading tournament…")
tournament_data = load_tournament_data(ds_cfg, feat_cfg, feature_stats=feature_stats)
log.info("tournament: N=%d input_dim=%d bins=%s",
         tournament_data.X.shape[0], tournament_data.input_dim,
         tournament_data.bin_names)
log.info("tournament label distribution: %s",
         pd.Series(tournament_data.y).value_counts().to_dict())
assert tournament_data.input_dim == input_dim

In [ ]:
# ------------------------------------------------------------
# Cell 4. Re-run inference (model) and build simple-detector predictions.
# ------------------------------------------------------------

CHEAT_TAG = "ALL"

log.info("Re-running model on synth test…")
synth_results, synth_preds = evaluate_on_synth_test(
    synth_data=synth_data,
    cheat_name=CHEAT_TAG,
    model_path=source_model_path,
    model_config=model_cfg,
    threshold=threshold,
    input_dim=input_dim,
    device=device,
    batch_size=4096,
    out_dir=None,
    return_predictions=True,
)

log.info("Re-running model on tournament…")
tourn_results, tourn_preds = evaluate_on_tournament(
    tournament_data=tournament_data,
    model_path=source_model_path,
    model_config=model_cfg,
    threshold=threshold,
    input_dim=input_dim,
    device=device,
    batch_size=4096,
    out_dir=None,
    cheat_name=CHEAT_TAG,
    return_predictions=True,
)

# --- rename model prediction columns; add simple-detector predictions ---
for df in (synth_preds, tourn_preds):
    df.rename(columns={
        "logit": "logit_model",
        "prob":  "prob_model",
        "pred":  "pred_model",
    }, inplace=True)
    # simple detector: predict 1 iff the move played equals sf15 top-1.
    sf = df["sf15_match"].astype(np.int8)
    df["prob_simple"] = sf.astype(np.float32)
    df["pred_simple"] = sf.astype(np.int8)

log.info("synth_preds shape=%s  tourn_preds shape=%s",
         synth_preds.shape, tourn_preds.shape)

# --- headline metrics (recomputed here so log has a single source of truth) ---
def _headline(df, label):
    y = df["y"].to_numpy()
    mm = compute_metrics(y, df["pred_model"].to_numpy())
    ms = compute_metrics(y, df["pred_simple"].to_numpy())
    log.info("[%s | model ] macro_f1=%.4f  cheat_P=%.4f  cheat_R=%.4f  cheat_F1=%.4f  acc=%.4f",
             label, mm["macro_f1"], mm["cheat_precision"], mm["cheat_recall"],
             mm["cheat_f1"], mm["accuracy"])
    log.info("[%s | simple] macro_f1=%.4f  cheat_P=%.4f  cheat_R=%.4f  cheat_F1=%.4f  acc=%.4f",
             label, ms["macro_f1"], ms["cheat_precision"], ms["cheat_recall"],
             ms["cheat_f1"], ms["accuracy"])
    return {"label": label, "model": mm, "simple": ms}

headline = [_headline(synth_preds, "synth_test"),
            _headline(tourn_preds, "tournament")]
# persist headline numbers as a flat CSV
rows = []
for h in headline:
    for det in ("model", "simple"):
        row = {"split": h["label"], "detector": det}
        row.update(h[det])
        rows.append(row)
pd.DataFrame(rows).to_csv(run_dir / "tables" / "headline_metrics.csv", index=False)

In [ ]:
# ------------------------------------------------------------
# Cell 5. Annotate tournament predictions with cheat-run info.
#   Guarantee from user: within one game_id, rows are contiguous (CSV order).
#   We additionally sort by (game_id, half_move) to be robust.
# ------------------------------------------------------------

def annotate_cheat_runs(df: pd.DataFrame) -> pd.DataFrame:
    """Add cheat_run_id / cheat_run_pos / cheat_run_len / cheat_order_in_game /
    game_total_cheats / game_total_moves. Rows reordered by (game_id, half_move)."""
    d = df.sort_values(["game_id", "half_move"]).reset_index(drop=True).copy()
    y = d["y"].astype(int).to_numpy()
    game = d["game_id"].to_numpy()
    prev_y = np.r_[0, y[:-1]]
    prev_game = np.r_[np.array(["__NONE__"], dtype=object), game[:-1]]
    new_run_start = (y == 1) & ((prev_y == 0) | (prev_game != game))
    # cheat_run_id: cumulative sum of run starts, 0 on non-cheat rows
    run_id = np.cumsum(new_run_start).astype(np.int64)
    run_id[y == 0] = 0
    d["cheat_run_id"] = run_id
    # position within run (1-indexed), 0 on non-cheat
    pos = np.zeros(len(d), dtype=np.int64)
    current_pos = 0
    current_run = 0
    for i in range(len(d)):
        if y[i] == 1:
            if run_id[i] != current_run:
                current_pos = 1
                current_run = run_id[i]
            else:
                current_pos += 1
            pos[i] = current_pos
        else:
            current_run = 0
            current_pos = 0
    d["cheat_run_pos"] = pos
    # run length: size of each run_id group (only for run_id > 0)
    run_sizes = pd.Series(run_id[run_id > 0]).value_counts().to_dict()
    d["cheat_run_len"] = np.array(
        [run_sizes.get(r, 0) for r in run_id], dtype=np.int64
    )
    # order of cheat within game (1-indexed among cheats only, 0 on non-cheat)
    order = d.groupby("game_id")["y"].cumsum().to_numpy(dtype=np.int64)
    order = np.where(y == 1, order, 0)
    d["cheat_order_in_game"] = order
    # game-level counters
    d["game_total_cheats"] = d.groupby("game_id")["y"].transform("sum").astype(int)
    d["game_total_moves"] = d.groupby("game_id")["y"].transform("size").astype(int)
    return d

tourn_enriched = annotate_cheat_runs(tourn_preds)

# quick diagnostics
n_games = tourn_enriched["game_id"].nunique()
n_cheat = int((tourn_enriched["y"] == 1).sum())
n_runs = int(tourn_enriched["cheat_run_id"].max()) if len(tourn_enriched) else 0
log.info("tournament: games=%d  rows=%d  cheat_moves=%d  cheat_runs=%d",
         n_games, len(tourn_enriched), n_cheat, n_runs)
runs_only = tourn_enriched.loc[tourn_enriched["cheat_run_id"] > 0, "cheat_run_id"]
run_len_series = runs_only.value_counts()
log.info("run length histogram: %s",
         run_len_series.value_counts().sort_index().to_dict())
log.info("games with run_len>=2: %d",
         int((tourn_enriched["cheat_run_len"] >= 2).groupby(tourn_enriched["game_id"]).any().sum()))
log.info("games with run_len>=3: %d",
         int((tourn_enriched["cheat_run_len"] >= 3).groupby(tourn_enriched["game_id"]).any().sum()))

In [ ]:
# ------------------------------------------------------------
# Cell 6. Q1 stats: model vs sf15-baseline disagreement & per-source recall.
# ------------------------------------------------------------

def q1_contingency(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """2×2×2 contingency (y, pred_model, pred_simple). Returns flat DataFrame."""
    ct = (df.groupby(["y", "pred_model", "pred_simple"]).size()
            .rename("count").reset_index())
    ct["split"] = label
    ct["frac"] = ct["count"] / len(df)
    return ct

def q1_disagreement_summary(df: pd.DataFrame, label: str) -> None:
    pm = df["pred_model"].to_numpy()
    ps = df["pred_simple"].to_numpy()
    y = df["y"].to_numpy()
    agree = (pm == ps).mean()
    log.info("[%s] overall model<->simple agreement: %.4f  (disagree=%.4f)",
             label, agree, 1 - agree)
    for y_val, y_name in [(0, "human"), (1, "cheat")]:
        sub = df[df["y"] == y_val]
        if len(sub) == 0:
            continue
        n = len(sub)
        n_mc_only  = int(((sub["pred_model"] == 1) & (sub["pred_simple"] == 0)).sum())
        n_sc_only  = int(((sub["pred_model"] == 0) & (sub["pred_simple"] == 1)).sum())
        n_both_c   = int(((sub["pred_model"] == 1) & (sub["pred_simple"] == 1)).sum())
        n_both_h   = int(((sub["pred_model"] == 0) & (sub["pred_simple"] == 0)).sum())
        log.info("  [%s | y=%s N=%d] both-cheat=%d both-fair=%d "
                 "model-only-cheat=%d simple-only-cheat=%d",
                 label, y_name, n, n_both_c, n_both_h, n_mc_only, n_sc_only)

def q1_synth_conditionals(df: pd.DataFrame) -> None:
    # (A) Cheats where sf15_match == 0 → baseline misses by construction;
    #     model recall here is evidence of non-artifact signal.
    cheat_no_sf15 = df[(df["y"] == 1) & (df["sf15_match"] == 0)]
    if len(cheat_no_sf15) > 0:
        recall = float(cheat_no_sf15["pred_model"].mean())
        log.info("[synth] CHEAT with sf15_match=0 (baseline-miss): "
                 "model recall=%.4f  N=%d", recall, len(cheat_no_sf15))
    # (B) Humans where sf15_match == 1 → baseline false-positive; what model says.
    human_sf15 = df[(df["y"] == 0) & (df["sf15_match"] == 1)]
    if len(human_sf15) > 0:
        fp = float(human_sf15["pred_model"].mean())
        log.info("[synth] HUMAN with sf15_match=1 (baseline-FP): "
                 "model FP-rate=%.4f  N=%d", fp, len(human_sf15))
    # (C) Per-cheat-source recall on synth cheats.
    log.info("[synth] per-cheat-source recall:")
    src_rows = []
    for src, sub in df[df["y"] == 1].groupby("cheat_source"):
        rec_m = float(sub["pred_model"].mean())
        rec_s = float(sub["pred_simple"].mean())
        sf_rate = float(sub["sf15_match"].mean())
        log.info("  src=%-14s N=%-6d sf15_match_rate=%.4f  model_recall=%.4f  simple_recall=%.4f",
                 src, len(sub), sf_rate, rec_m, rec_s)
        src_rows.append({
            "cheat_source": src, "n_cheat": len(sub),
            "sf15_match_rate": sf_rate,
            "model_recall": rec_m, "simple_recall": rec_s,
        })
    pd.DataFrame(src_rows).to_csv(
        run_dir / "tables" / "synth_per_cheat_source.csv", index=False
    )

for df, label in [(synth_preds, "synth_test"), (tourn_enriched, "tournament")]:
    q1_disagreement_summary(df, label)
    ct = q1_contingency(df, label)
    ct.to_csv(run_dir / "tables" / f"q1_contingency_{label}.csv", index=False)
    log.info("[%s] contingency saved (%d rows)", label, len(ct))

q1_synth_conditionals(synth_preds)

In [ ]:
# ------------------------------------------------------------
# Cell 7. Q2 stats: consecutive-cheat detection on tournament.
# ------------------------------------------------------------

cheats = tourn_enriched[tourn_enriched["y"] == 1].copy()
log.info("[tournament] total cheat moves: %d  (in %d cheat runs)",
         len(cheats), int(cheats["cheat_run_id"].nunique()))

def _bucket_stats(cheats: pd.DataFrame, bucket_col: str,
                  buckets: list, last_ge: int | None = None,
                  out_name: str = "") -> pd.DataFrame:
    """For each bucket value, report N, model_recall, simple_recall."""
    rows = []
    for b in buckets:
        sub = cheats[cheats[bucket_col] == b]
        if len(sub) == 0:
            continue
        rows.append({
            bucket_col: str(b),
            "n": len(sub),
            "model_recall": float(sub["pred_model"].mean()),
            "simple_recall": float(sub["pred_simple"].mean()),
            "sf15_match_rate": float(sub["sf15_match"].mean()),
        })
    if last_ge is not None:
        sub = cheats[cheats[bucket_col] >= last_ge]
        if len(sub) > 0:
            rows.append({
                bucket_col: f">={last_ge}",
                "n": len(sub),
                "model_recall": float(sub["pred_model"].mean()),
                "simple_recall": float(sub["pred_simple"].mean()),
                "sf15_match_rate": float(sub["sf15_match"].mean()),
            })
    tbl = pd.DataFrame(rows)
    log.info("[tournament] recall by %s:\n%s",
             bucket_col, tbl.to_string(index=False))
    if out_name:
        tbl.to_csv(run_dir / "tables" / out_name, index=False)
    return tbl

# Position within a consecutive cheat run (1st, 2nd, 3rd, >=4)
_bucket_stats(cheats, "cheat_run_pos", [1, 2, 3], last_ge=4,
              out_name="q2_recall_by_run_pos.csv")
# Absolute order among all cheats in the game
_bucket_stats(cheats, "cheat_order_in_game", [1, 2, 3, 4, 5], last_ge=6,
              out_name="q2_recall_by_order_in_game.csv")
# Run length categories
_bucket_stats(cheats, "cheat_run_len", [1, 2, 3], last_ge=4,
              out_name="q2_recall_by_run_len.csv")
# Total cheats in the game
_bucket_stats(cheats, "game_total_cheats", [1, 2, 3, 4, 5], last_ge=6,
              out_name="q2_recall_by_game_total_cheats.csv")

In [ ]:
# ------------------------------------------------------------
# Cell 8. Save enriched predictions + pre-filtered example subsets.
# ------------------------------------------------------------

# full enriched predictions
synth_out = run_dir / "predictions" / "synth_test.parquet"
tourn_out = run_dir / "predictions" / "tournament.parquet"
synth_preds.to_parquet(synth_out, index=False)
tourn_enriched.to_parquet(tourn_out, index=False)
log.info("synth predictions saved  -> %s (shape=%s)", synth_out, synth_preds.shape)
log.info("tourn predictions saved  -> %s (shape=%s)", tourn_out, tourn_enriched.shape)

def _save_examples(df: pd.DataFrame, mask, name: str, top: int | None = None):
    sub = df[mask].copy()
    # optional sort key: largest |prob_model - pred_simple| first to surface the
    # strongest disagreement cases
    sub["__delta"] = (sub["prob_model"] - sub["prob_simple"]).abs()
    sub = sub.sort_values("__delta", ascending=False).drop(columns=["__delta"])
    if top is not None:
        sub = sub.head(top)
    out = run_dir / "examples" / f"{name}.parquet"
    sub.to_parquet(out, index=False)
    log.info("examples %-40s N=%-6d -> %s", name, len(sub), out)
    return sub

# ---- Q1 examples (tournament) ----
# model catches a cheat that is NOT sf15 top-1 → evidence against artifact.
_save_examples(
    tourn_enriched,
    (tourn_enriched["y"] == 1) & (tourn_enriched["sf15_match"] == 0) & (tourn_enriched["pred_model"] == 1),
    name="q1_model_catches_non_sf15_cheat",
)
# model ignores a human move that IS sf15 top-1 → evidence against artifact.
_save_examples(
    tourn_enriched,
    (tourn_enriched["y"] == 0) & (tourn_enriched["sf15_match"] == 1) & (tourn_enriched["pred_model"] == 0),
    name="q1_model_ignores_human_on_sf15_topline",
)
# same categories on synth-test
_save_examples(
    synth_preds,
    (synth_preds["y"] == 1) & (synth_preds["sf15_match"] == 0) & (synth_preds["pred_model"] == 1),
    name="q1_synth_model_catches_non_sf15_cheat",
)
_save_examples(
    synth_preds,
    (synth_preds["y"] == 0) & (synth_preds["sf15_match"] == 1) & (synth_preds["pred_model"] == 0),
    name="q1_synth_model_ignores_human_on_sf15_topline",
)

# ---- Q2 examples (tournament only) ----
# caught cheat in position >=2 of a consecutive run
_save_examples(
    tourn_enriched,
    (tourn_enriched["y"] == 1) & (tourn_enriched["cheat_run_pos"] >= 2) & (tourn_enriched["pred_model"] == 1),
    name="q2_model_catches_cheat_in_run_pos_ge2",
)
# full cheat run caught (every position in the run has pred_model=1)
run_all_caught = (
    tourn_enriched[tourn_enriched["cheat_run_id"] > 0]
    .groupby("cheat_run_id")
    .agg(all_caught=("pred_model", "all"), run_len=("cheat_run_len", "first"))
)
good_run_ids = run_all_caught[(run_all_caught["all_caught"]) & (run_all_caught["run_len"] >= 2)].index
_save_examples(
    tourn_enriched,
    tourn_enriched["cheat_run_id"].isin(good_run_ids),
    name="q2_full_runs_caught_len_ge2",
)
# cheat runs of length >=2 where the model missed at least one – failure analysis
missed_in_run = run_all_caught[(~run_all_caught["all_caught"]) & (run_all_caught["run_len"] >= 2)].index
_save_examples(
    tourn_enriched,
    tourn_enriched["cheat_run_id"].isin(missed_in_run),
    name="q2_runs_with_missed_cheat_len_ge2",
)

# ---- General failure buckets ----
_save_examples(
    tourn_enriched,
    (tourn_enriched["y"] == 1) & (tourn_enriched["pred_model"] == 0),
    name="misses_model_on_cheats",
)
_save_examples(
    tourn_enriched,
    (tourn_enriched["y"] == 0) & (tourn_enriched["pred_model"] == 1),
    name="false_positives_model_on_humans",
)

In [ ]:
# ------------------------------------------------------------
# Cell 9. Summary.
# ------------------------------------------------------------

def _fmt(v):
    try:
        return f"{float(v):.4f}"
    except Exception:
        return str(v)

log.info("=" * 78)
log.info("CASE STUDY SUMMARY (step 1 — predictions & stats only)")

m_synth = compute_metrics(synth_preds["y"].to_numpy(), synth_preds["pred_model"].to_numpy())
s_synth = compute_metrics(synth_preds["y"].to_numpy(), synth_preds["pred_simple"].to_numpy())
m_tourn = compute_metrics(tourn_enriched["y"].to_numpy(), tourn_enriched["pred_model"].to_numpy())
s_tourn = compute_metrics(tourn_enriched["y"].to_numpy(), tourn_enriched["pred_simple"].to_numpy())
log.info("headline |  split       | model macro_f1 | simple macro_f1 | model cheat_f1 | simple cheat_f1")
log.info("         |  synth_test  |  %s       |  %s        |  %s       |  %s",
         _fmt(m_synth["macro_f1"]), _fmt(s_synth["macro_f1"]),
         _fmt(m_synth["cheat_f1"]), _fmt(s_synth["cheat_f1"]))
log.info("         |  tournament  |  %s       |  %s        |  %s       |  %s",
         _fmt(m_tourn["macro_f1"]), _fmt(s_tourn["macro_f1"]),
         _fmt(m_tourn["cheat_f1"]), _fmt(s_tourn["cheat_f1"]))

log.info("artefacts in %s:", run_dir)
for p in sorted(run_dir.rglob("*")):
    if p.is_file():
        try:
            sz = p.stat().st_size
        except OSError:
            sz = -1
        rel = p.relative_to(run_dir)
        log.info("  %8d bytes  %s", sz, rel)
log.info("=" * 78)
log.info("DONE at %s", datetime.now().isoformat(timespec="seconds"))